## Restaurant Discovery using OpenStreetMap Overpass API

imports

In [1]:
import overpy
from geopy.geocoders import Nominatim

# Initialize the geolocator with a custom user agent name
geolocator = Nominatim(user_agent="FedUp")

# Convert a human-readable address into exact coordinates
location = geolocator.geocode("715 E 35th Street, Charlotte, NC")

print(f"Address: {location.address}")
print(f"Latitude: {location.latitude}, Longitude: {location.longitude}")


Address: 715, East 35th Street, Shamrock Green, NoDa, Charlotte, Mecklenburg County, North Carolina, 28205, United States
Latitude: 35.2455662, Longitude: -80.8037229


In [2]:
import requests

lat = 35.2455
lon = -80.8037
radius = 1500  # meters

query = f"""
[out:json][timeout:25];
(
  node["amenity"="restaurant"](around:{radius},{lat},{lon});
  node["amenity"="bar"](around:{radius},{lat},{lon});
  node["amenity"="pub"](around:{radius},{lat},{lon});
  node["amenity"="fast_food"](around:{radius},{lat},{lon});
  node["amenity"="cafe"](around:{radius},{lat},{lon});
);
out body;
"""

response = requests.post(
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    data={"data": query},
    headers={"User-Agent": "FedUp/1.0"},
    timeout=30
)
response.raise_for_status()
data = response.json()

for element in data["elements"]:
    tags = element.get("tags", {})
    print(element["id"], tags.get("name"), tags.get("amenity"), tags.get("cuisine"), element["lat"], element["lon"])

2990864745 Cabo Fish Taco restaurant None 35.2471262 -80.8057495
5683138346 Boudreaux’s Louisiana Kitchen restaurant None 35.2476807 -80.8042851
6257045702 Benny Pennello’s restaurant pizza 35.2453529 -80.8092233
6532532816 Chinese Fast Food restaurant chinese 35.247198 -80.8185552
6532532844 Popeyes fast_food chicken 35.256581 -80.7971833
6585332735 Negril Social restaurant None 35.253862 -80.8053345
9222745717 The Hobbyist cafe coffee_shop 35.2386029 -80.8160683
10802248505 Salud Cervezeria restaurant latin_food 35.2478558 -80.8041412
10874217455 Sabor Latin Street Grill restaurant mexican 35.2472233 -80.8055827
11054065701 Bop Bop fast_food korean 35.2406306 -80.814397
11054065703 UDON restaurant japanese 35.2407065 -80.8142923
11054065704 Stuffed fast_food dumplings 35.2406744 -80.814339
11054082605 Labaratory bar None 35.2408511 -80.8145595
11054082607 & Coffee cafe coffee_shop 35.2405408 -80.8145294
11054082608 The Rare Butcher fast_food steak_house 35.2409107 -80.8144022
1105408

In [3]:
import sqlite3

conn = sqlite3.connect("fedup.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS places (
        id INTEGER PRIMARY KEY,
        name TEXT,
        amenity TEXT,
        cuisine TEXT,
        lat REAL,
        lon REAL
    )
""")

for element in data["elements"]:
    tags = element.get("tags", {})
    cursor.execute("""
        INSERT OR REPLACE INTO places (id, name, amenity, cuisine, lat, lon)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        element["id"],
        tags.get("name"),
        tags.get("amenity"),
        tags.get("cuisine"),
        element["lat"],
        element["lon"]
    ))

conn.commit()
conn.close()
print(f"Saved {len(data['elements'])} places to fedup.db")

Saved 47 places to fedup.db
